In [ ]:
# Install the project dependencies required by this notebook.
%pip install -q pandas numpy duckdb pyarrow scikit-learn xgboost mlflow matplotlib scipy joblib pyyaml

In [ ]:
# Mount Google Drive so Colab can access the private MIMIC-IV files and derived artifacts.
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# Clone or update the GitHub repository so the notebook can import the shared project code.
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/mbakos95/aki-sentinel.git"
REPO_DIR = Path("/content/aki-sentinel")

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=True)

sys.path.insert(0, str(REPO_DIR))

In [ ]:
# Define the private MIMIC-IV and artifact locations used by the pipeline.
from pathlib import Path

MIMIC_ROOT = Path("/content/drive/MyDrive/MIMIC-IV")
HOSP_DIR = MIMIC_ROOT / "hosp"
ICU_DIR = MIMIC_ROOT / "icu"

PRIVATE_ROOT = Path("/content/drive/MyDrive/AKI-Sentinel-Private")
ARTIFACT_DIR = PRIVATE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Import the cohort and feature-building functions used to create the 12-hour ML dataset.
import pandas as pd

from src.cohort import build_eligible_cohort
from src.features import build_feature_table

In [ ]:
# Load the eligible cohort and AKI labels produced by the previous notebook.
COHORT_PATH = ARTIFACT_DIR / "eligible_cohort.parquet"
LABEL_PATH = ARTIFACT_DIR / "labels" / "aki_labels.parquet"

eligible_cohort = (
    pd.read_parquet(COHORT_PATH)
    if COHORT_PATH.exists()
    else build_eligible_cohort(HOSP_DIR, ICU_DIR, observation_hours=12)
)
labels = pd.read_parquet(LABEL_PATH)

In [ ]:
# Build demographics, laboratory, vital-sign, weight, and urine-output features using only the first 12 ICU hours.
ml_dataset = build_feature_table(
    cohort=eligible_cohort,
    labels=labels,
    labevents_path=HOSP_DIR / "labevents.csv.gz",
    chartevents_path=ICU_DIR / "chartevents.csv.gz",
    outputevents_path=ICU_DIR / "outputevents.csv.gz",
)

In [ ]:
# Inspect the resulting one-row-per-ICU-stay modeling table and its target prevalence.
print("Rows:", f"{len(ml_dataset):,}")
print("Columns:", ml_dataset.shape[1])
print("Positive rate:", f"{ml_dataset['target'].mean():.3%}")
ml_dataset.head()

In [ ]:
# Inspect feature missingness before model-side imputation is applied.
missingness = (
    ml_dataset.isna()
    .mean()
    .sort_values(ascending=False)
    .rename("missing_rate")
    .to_frame()
)
missingness.head(25)

In [ ]:
# Save the private feature table that will be used for model training and evaluation.
FEATURE_DIR = ARTIFACT_DIR / "features"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
ml_dataset.to_parquet(FEATURE_DIR / "ml_dataset.parquet", index=False)